# vF.0.4 FakeShield-Lite v0.4 — Baseline
# Tampered Image Detection & Localization

---

**Assignment:** Big Vision Internship — Tampered Image Detection & Localization 
**Model:** FakeShield-Lite (pruned from FakeShield, Xu et al., ICLR 2025) 
**Runtime:** Kaggle Notebook — GPU T4 (16 GB VRAM) 
**Dataset:** [CASIA Splicing Detection + Localization](https://www.kaggle.com/datasets/sagnikkayalcse52/casia-spicing-detection-localization) (connected as Kaggle input) 
**Version:** vF.0.4 — Kaggle-native baseline (SAM batch-decode fix)

---

## Section 1 — Environment Setup

In [ ]:
# ============================================================================
# 1.1 Install Dependencies
# ============================================================================
# PyTorch, torchvision, and transformers are pre-installed on Kaggle.
# We only install packages not in the default Kaggle environment.
!pip install -q segment-anything
!pip install -q albumentations>=1.3.1

In [ ]:
# ============================================================================
# 1.2 GPU Check
# ============================================================================
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM            : {vram:.1f} GB")
else:
    print("WARNING: No GPU detected. Enable GPU via Settings > Accelerator > GPU T4 x2.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print()
print(f"Using device    : {DEVICE}")

In [ ]:
# ============================================================================
# 1.3 Common Imports & Reproducibility
# ============================================================================
import os
import cv2
import glob
import random
import numpy as np
from PIL import Image
from io import BytesIO
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print("All imports loaded. Seed set to", SEED)

---
## Section 2 — Problem Overview

### What is Image Tampering?

Image tampering refers to any deliberate modification of an image to change its content. Common techniques include:

| Technique | Description |
|---|---|
| **Copy-Move** | A region is copied and pasted within the same image |
| **Splicing** | Part of one image is pasted into another image |
| **Removal / Inpainting** | An object is removed and the gap is filled |

### Why Detection + Localization?

Simple detection (real / fake) is not enough. Knowing **where** the tampered region is provides:
- Evidence for forensic analysis
- Interpretability of the model decision
- Basis for downstream verification

### Our Approach: FakeShield-Lite

We adapt **FakeShield** (Xu et al., ICLR 2025), a state-of-the-art multi-modal framework for
explainable image forgery detection and localization. The full FakeShield uses two 13B-parameter
LLMs, demanding ~50 GB VRAM. We prune it to **FakeShield-Lite** (~182M parameters)
that fits on a Colab T4 while preserving the core architectural ideas:

- **Dual-encoder design** (CLIP + SAM) from FakeShield’s DTE-FDM and MFLM modules
- **SAM-based mask generation** with learned visual prompts
- **Combined detection + localization** with Dice + BCE loss (FakeShield Eq. 5)

---
## Section 3 — Dataset Setup

### 3.1 Dataset Source

We use the **CASIA Splicing Detection + Localization** dataset from Kaggle, connected
directly as a Kaggle input.

**Actual dataset structure on Kaggle:**
```
casia-spicing-detection-localization/
  New folder/
    IMAGE/
      Au/     # Authentic images
      Tp/     # Tampered images
    MASK/
      Au/     # Masks for authentic (all black)
      Tp/     # Ground-truth masks for tampered images
```

Images and masks are paired by **matching filenames** across `IMAGE/` and `MASK/`.

In [ ]:
# ============================================================================
# 3.1 Connect to Kaggle Dataset
# ============================================================================
# The dataset slug on disk may differ from the URL slug.
# Instead of hardcoding, we search /kaggle/input/ for Au/ + Tp/ directories.

from pathlib import Path

def find_dataset():
    """Search /kaggle/input/ for Au/ and Tp/ directories.

    Returns: (au_dir, tp_dir, mask_tp_dir_or_None)
    """
    search_base = "/kaggle/input"
    candidates = []  # (dirpath, au, tp)

    for dirpath, dirnames, _ in os.walk(search_base):
        if "Au" in dirnames and "Tp" in dirnames:
            candidates.append((
                dirpath,
                os.path.join(dirpath, "Au"),
                os.path.join(dirpath, "Tp"),
            ))

    if not candidates:
        # Print tree for debugging
        print("Could not find Au/ + Tp/.  Contents of /kaggle/input/:")
        for dirpath, dirnames, _ in os.walk(search_base):
            depth = dirpath.replace(search_base, "").count(os.sep)
            print(f"{'  ' * depth}{os.path.basename(dirpath)}/")
            if depth >= 3:
                break
        raise FileNotFoundError("No Au/ + Tp/ found under " + search_base)

    # Separate IMAGE dirs from MASK dirs
    image_cands = [c for c in candidates if "mask" not in c[0].lower()]
    mask_cands  = [c for c in candidates if "mask" in c[0].lower()]

    # Prefer the one with 'image' in path, else first non-mask candidate
    if image_cands:
        explicit = [c for c in image_cands if "image" in c[0].lower()]
        chosen = explicit[0] if explicit else image_cands[0]
    else:
        chosen = candidates[0]

    # Detect MASK/Tp/ for ground-truth masks
    mask_tp_dir = None
    if mask_cands:
        cand_tp = os.path.join(mask_cands[0][0], "Tp")
        if os.path.isdir(cand_tp):
            mask_tp_dir = cand_tp

    return chosen[1], chosen[2], mask_tp_dir


AU_DIR, TP_DIR, MASK_TP_DIR = find_dataset()

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

au_count = len([f for f in os.listdir(AU_DIR) if Path(f).suffix.lower() in IMG_EXTS])
tp_count = len([f for f in os.listdir(TP_DIR) if Path(f).suffix.lower() in IMG_EXTS])

print(f"Authentic dir : {AU_DIR}  ({au_count} images)")
print(f"Tampered dir  : {TP_DIR}  ({tp_count} images)")
if MASK_TP_DIR:
    mk_count = len([f for f in os.listdir(MASK_TP_DIR) if Path(f).suffix.lower() in IMG_EXTS])
    print(f"Mask dir (Tp) : {MASK_TP_DIR}  ({mk_count} masks)")
else:
    print("Mask dir (Tp) : NOT FOUND")

In [ ]:
# ============================================================================
# 3.2 Discover & Pair Images with Masks
# ============================================================================

def find_images(directory) -> List[str]:
    """Find all image files in a directory (non-recursive)."""
    images = []
    for f in sorted(os.listdir(directory)):
        if Path(f).suffix.lower() in IMG_EXTS:
            images.append(os.path.join(directory, f))
    return images


def find_mask_for_image(image_path: str, mask_dir: str) -> Optional[str]:
    """Find matching mask by filename stem in the mask directory."""
    if mask_dir is None:
        return None
    stem = Path(image_path).stem
    for ext in [".png", ".bmp", ".jpg", ".tif"]:
        p = os.path.join(mask_dir, stem + ext)
        if os.path.exists(p):
            return p
    # Try _gt suffix
    for ext in [".png", ".bmp", ".jpg", ".tif"]:
        p = os.path.join(mask_dir, stem + "_gt" + ext)
        if os.path.exists(p):
            return p
    return None


# --- Authentic images ---
au_images = find_images(AU_DIR)

# --- Tampered images + mask pairing ---
tp_images = find_images(TP_DIR)

tp_paired = []     # (image_path, mask_path)
tp_no_mask = []
for img_path in tp_images:
    m = find_mask_for_image(img_path, MASK_TP_DIR)
    if m:
        tp_paired.append((img_path, m))
    else:
        tp_no_mask.append(img_path)

print(f"Authentic images      : {len(au_images)}")
print(f"Tampered images       : {len(tp_images)}")
print(f"  with mask           : {len(tp_paired)}")
print(f"  without mask (skip) : {len(tp_no_mask)}")

# Show a few filename examples
if tp_paired:
    print()
    print("Example pairs:")
    for img, msk in tp_paired[:3]:
        print(f"  img: {Path(img).name}")
        print(f"  msk: {Path(msk).name}")
        print()

In [ ]:
# ============================================================================
# 3.3 Build Unified Sample List & Split
# ============================================================================

# Each sample: (image_path, mask_path_or_None, label)
#   label: 0 = authentic, 1 = tampered

all_samples = []

for img in au_images:
    all_samples.append((img, None, 0))

for img, msk in tp_paired:
    all_samples.append((img, msk, 1))

random.shuffle(all_samples)

# 80 / 10 / 10 split
n = len(all_samples)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_samples = all_samples[:n_train]
val_samples   = all_samples[n_train:n_train + n_val]
test_samples  = all_samples[n_train + n_val:]

def count_labels(samples):
    c = defaultdict(int)
    for _, _, lbl in samples:
        c[lbl] += 1
    return dict(c)

print(f"Total samples : {n}")
print(f"Train         : {len(train_samples)}  {count_labels(train_samples)}")
print(f"Val           : {len(val_samples)}  {count_labels(val_samples)}")
print(f"Test          : {len(test_samples)}  {count_labels(test_samples)}")

In [ ]:
# ============================================================================
# 3.4 Visualize Dataset Samples
# ============================================================================

def show_samples(samples, title="Dataset Samples", n=4):
    """Display original images alongside their ground-truth masks."""
    tampered = [s for s in samples if s[2] == 1 and s[1] is not None]
    picks = random.sample(tampered, min(n, len(tampered)))

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    axes[0, 0].set_title("Original Image", fontsize=13, fontweight="bold")
    axes[0, 1].set_title("Ground Truth Mask", fontsize=13, fontweight="bold")
    axes[0, 2].set_title("Overlay", fontsize=13, fontweight="bold")

    for i, (img_path, msk_path, lbl) in enumerate(picks):
        img = np.array(Image.open(img_path).convert("RGB"))
        msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        msk = (msk > 127).astype(np.float32)

        # resize for display
        h, w = 256, 256
        img_r = cv2.resize(img, (w, h))
        msk_r = cv2.resize(msk, (w, h), interpolation=cv2.INTER_NEAREST)

        overlay = img_r.copy().astype(np.float32)
        red = np.zeros_like(overlay)
        red[:, :, 0] = 255
        m3 = np.stack([msk_r]*3, axis=-1)
        overlay = overlay * (1 - 0.4 * m3) + red * (0.4 * m3)
        overlay = np.clip(overlay, 0, 255).astype(np.uint8)

        axes[i, 0].imshow(img_r);  axes[i, 0].axis("off")
        axes[i, 1].imshow(msk_r, cmap="hot", vmin=0, vmax=1); axes[i, 1].axis("off")
        axes[i, 2].imshow(overlay); axes[i, 2].axis("off")

    plt.suptitle(title, fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()

show_samples(train_samples, title="Training Set Samples")

---
## Section 4 — Data Augmentation

We use **Albumentations** for training augmentation. Augmentations are applied identically
to both the image and its mask so that spatial correspondence is preserved.

| Augmentation | Purpose |
|---|---|
| HorizontalFlip | Spatial invariance |
| RandomRotate90 | Orientation invariance |
| ColorJitter | Illumination robustness |
| GaussNoise | Sensor noise robustness |
| ImageCompression | JPEG robustness (bonus) |
| Resize(256) | Fixed input size |

In [ ]:
# ============================================================================
# 4.1 Define Augmentation Transforms
# ============================================================================

IMG_SIZE = 256

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),  # JPEG robustness
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print("Transforms defined.")
print(f"  Train augmentations: {len(train_transform.transforms)} ops")
print(f"  Val   augmentations: {len(val_transform.transforms)} ops")

In [ ]:
# ============================================================================
# 4.2 Visualize Augmentation Examples
# ============================================================================

# Pick one tampered sample
demo_samples = [s for s in train_samples if s[2] == 1 and s[1] is not None]
demo_img_path, demo_msk_path, _ = demo_samples[0]
demo_img = np.array(Image.open(demo_img_path).convert("RGB"))
demo_msk = cv2.imread(demo_msk_path, cv2.IMREAD_GRAYSCALE)
demo_msk = (demo_msk > 127).astype(np.uint8)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Show original in first column
r = A.Resize(IMG_SIZE, IMG_SIZE)(image=demo_img, mask=demo_msk)
axes[0, 0].imshow(r["image"]); axes[0, 0].set_title("Original"); axes[0, 0].axis("off")
axes[1, 0].imshow(r["mask"], cmap="hot"); axes[1, 0].set_title("Mask"); axes[1, 0].axis("off")
axes[2, 0].axis("off")

# Augmented versions
aug_no_norm = A.Compose([  # without normalize for visualization
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.8),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.5),
    A.ImageCompression(quality_lower=70, quality_upper=100, p=0.5),
])

for col in range(1, 4):
    result = aug_no_norm(image=demo_img, mask=demo_msk)
    axes[0, col].imshow(result["image"])
    axes[0, col].set_title(f"Augmented #{col}")
    axes[0, col].axis("off")
    axes[1, col].imshow(result["mask"], cmap="hot", vmin=0, vmax=1)
    axes[1, col].axis("off")
    axes[2, col].axis("off")

plt.suptitle("Data Augmentation Examples", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 4.3 Dataset Class
# ============================================================================

class TamperDataset(Dataset):
    """
    Dataset for image tampering detection & localization.

    Each sample yields:
      image  : (3, H, W) float tensor, normalised
      mask   : (H, W) float tensor, 0 or 1
      label  : scalar float tensor, 0=authentic 1=tampered
    """

    def __init__(self, samples: list, transform=None):
        self.samples = samples       # [(img_path, mask_path_or_None, label), ...]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, msk_path, label = self.samples[idx]

        # Load image
        image = np.array(Image.open(img_path).convert("RGB"))

        # Load mask (zeros for authentic images)
        if msk_path is not None and os.path.exists(msk_path):
            mask = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros(image.shape[:2], dtype=np.uint8)
            else:
                mask = (mask > 127).astype(np.uint8)      # binarise
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)

        # Apply augmentations (same spatial transform for image + mask)
        if self.transform:
            result = self.transform(image=image, mask=mask)
            image = result["image"]                         # (3, H, W) float
            mask  = result["mask"].float()                  # (H, W) float {0,1}
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask  = torch.from_numpy(mask).float()

        return {
            "image": image,
            "mask": mask,
            "label": torch.tensor(label, dtype=torch.float32),
        }


# Build datasets
train_ds = TamperDataset(train_samples, transform=train_transform)
val_ds   = TamperDataset(val_samples,   transform=val_transform)
test_ds  = TamperDataset(test_samples,  transform=val_transform)

# DataLoaders
BATCH_SIZE = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Quick test
batch = next(iter(train_loader))
print(f"Batch shapes  —  image: {batch['image'].shape}  mask: {batch['mask'].shape}  label: {batch['label'].shape}")

---
## Section 5 — FakeShield-Lite Architecture

### 5.1 From FakeShield to FakeShield-Lite

FakeShield (Xu et al., ICLR 2025) uses:

| Component | Params | Role |
|---|---|---|
| CLIP ViT-L/14-336 | 304M | Visual feature encoder |
| LLaVA-v1.5-13B (DTE-FDM) | 13B | Detection + text explanation |
| Domain Tag Generator | 25M | Classifies tamper domain (PS / DF / AIGC) |
| Tamper Comprehension Module | 7–13B | Aligns text with visual features |
| SAM ViT-H | 636M | Segmentation encoder + decoder |
| **Total** | **~20–27B** | **Needs ~50 GB VRAM** |

**FakeShield-Lite** keeps the dual-encoder + SAM decoder pipeline but removes the LLMs:

| Component | Params | What changed |
|---|---|---|
| CLIP ViT-B/16 (frozen) | 86M | Downsized from ViT-L |
| SAM ViT-B encoder (frozen) | 91M | Downsized from ViT-H |
| SAM mask decoder (trainable) | 4M | Kept as-is |
| Detection Head (MLP) | 0.4M | Simplified binary DTG |
| Feature Projection (MLP) | 0.8M | Replaces TCM |
| **Total** | **~182M** | **~1.5 GB VRAM** |

### 5.2 Architecture Diagram

```
Input Image (3×256×256)
       │
       ├───────────────────────────┐
       │                           │
       ▼                           ▼
 SAM ViT-B Encoder        CLIP ViT-B/16
   (frozen, 91M)          (frozen, 86M)
       │                           │
   E_mid (64×64×256)         [CLS] (768)
       │                     ┌─────┴─────┐
       │                     │           │
       │              Detection     Feature
       │              Head (MLP)    Projection
       │              768→1         768→256
       │                     │           │
       │                real/fake    h_prompt
       │                     │           │
       │                     │     SAM Prompt
       │                     │     Encoder
       │                     │           │
       └─────────────────┬───────────┘
                         ▼
               SAM Mask Decoder (4M)
                         │
                         ▼
                 Predicted Mask (H×W)
```

In [ ]:
# ============================================================================
# 5.3 Download Pretrained SAM ViT-B Weights
# ============================================================================

SAM_CHECKPOINT = "/kaggle/working/sam_vit_b_01ec64.pth"

if not os.path.exists(SAM_CHECKPOINT):
    print("Downloading SAM ViT-B checkpoint...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth \
          -O {SAM_CHECKPOINT}
    print("Done.")
else:
    print(f"SAM checkpoint already present: {SAM_CHECKPOINT}")

print(f"  Size: {os.path.getsize(SAM_CHECKPOINT) / 1e6:.0f} MB")

In [ ]:
# ============================================================================
# 5.4 Model Components
# ============================================================================
# Each component maps directly to a FakeShield module (see Section 5.1 table).

from transformers import CLIPVisionModel, CLIPImageProcessor
from segment_anything import sam_model_registry


class DetectionHead(nn.Module):
    """
    Binary classifier: authentic (0) vs tampered (1).
    Simplified from FakeShield's Domain Tag Generator (DTG).
    DTG classified 3 tamper domains; we reduce to binary detection.
    """

    def __init__(self, in_dim: int = 768):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def forward(self, cls_token: torch.Tensor) -> torch.Tensor:
        return self.head(cls_token).squeeze(-1)            # (B,)


class FeatureProjection(nn.Module):
    """
    Projects CLIP features into SAM's prompt embedding space (dim 256).

    Replaces FakeShield's Tamper Comprehension Module (TCM).
    In FakeShield, TCM is an LLM that encodes long-text descriptions into
    a <SEG> token embedding for SAM. Here we directly project CLIP features.

    Architecture mirrors FakeShield's text_hidden_fcs in GLaMM.py:
        Linear -> ReLU -> Linear -> Dropout
    """

    def __init__(self, in_dim: int = 768, out_dim: int = 256):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, out_dim),
        )

    def forward(self, cls_token: torch.Tensor) -> torch.Tensor:
        return self.projection(cls_token)                  # (B, 256)


class SAMBackbone(nn.Module):
    """
    Wraps SAM ViT-B for the localization branch.

    Mirrors FakeShield's GLaMMBaseModel behaviour:
      - Freeze image_encoder  (FakeShield: _configure_grounding_encoder)
      - Freeze prompt_encoder
      - Train  mask_decoder   (FakeShield: _train_mask_decoder)
    """

    def __init__(self, checkpoint: str):
        super().__init__()
        sam = sam_model_registry["vit_b"](checkpoint=checkpoint)
        self.image_encoder  = sam.image_encoder
        self.prompt_encoder = sam.prompt_encoder
        self.mask_decoder   = sam.mask_decoder

        # Freeze encoder + prompt encoder
        for p in self.image_encoder.parameters():
            p.requires_grad = False
        for p in self.prompt_encoder.parameters():
            p.requires_grad = False

        # Train mask decoder (matching FakeShield)
        for p in self.mask_decoder.parameters():
            p.requires_grad = True

    @torch.no_grad()
    def encode_image(self, x: torch.Tensor) -> torch.Tensor:
        """Run frozen SAM image encoder."""
        return self.image_encoder(x)

    def decode_mask(
        self,
        image_embeddings: torch.Tensor,
        prompt_embedding: torch.Tensor,
    ) -> torch.Tensor:
        """
        Generate mask from SAM decoder, guided by learned prompt.

        SAM's mask_decoder internally uses repeat_interleave which assumes
        single-image input. We loop per-image to support batch processing.
        """
        # Dense "no mask" embedding from prompt encoder — shape (1, C, H, W)
        _, dense_embeddings = self.prompt_encoder(
            points=None, boxes=None, masks=None,
        )
        image_pe = self.prompt_encoder.get_dense_pe()  # (1, 256, 64, 64)

        B = image_embeddings.shape[0]
        masks = []
        for i in range(B):
            sparse_i = prompt_embedding[i : i + 1].unsqueeze(1)   # (1, 1, 256)
            low_res, _ = self.mask_decoder(
                image_embeddings=image_embeddings[i : i + 1],     # (1, C, H, W)
                image_pe=image_pe,
                sparse_prompt_embeddings=sparse_i,
                dense_prompt_embeddings=dense_embeddings,          # (1, C, H, W)
                multimask_output=False,
            )
            masks.append(low_res[:, 0])                            # (1, 256, 256)
        return torch.cat(masks, dim=0)                             # (B, 256, 256)


print("Components defined: DetectionHead, FeatureProjection, SAMBackbone")

In [ ]:
# ============================================================================
# 5.5 FakeShield-Lite — Full Model
# ============================================================================

class FakeShieldLite(nn.Module):
    """
    FakeShield-Lite: Pruned FakeShield for Colab T4.

    KEPT from FakeShield:
      * Dual encoder (CLIP + SAM)   ← DTE-FDM vision encoder + MFLM SAM
      * SAM mask decoder w/ prompt  ← MFLM mask generation
      * Dice + BCE mask loss        ← Eq. 5
    REMOVED:
      * LLaVA-13B  (DTE-FDM LLM)   ← cannot fit T4
      * TCM LLM    (MFLM encoder)   ← cannot fit T4
      * Text explanation pipeline   ← not required
    """

    # SAM image normalisation (from FakeShield / Meta)
    SAM_MEAN = torch.tensor([123.675, 116.28, 103.53]).view(1, 3, 1, 1)
    SAM_STD  = torch.tensor([58.395,  57.12,  57.375]).view(1, 3, 1, 1)

    def __init__(
        self,
        clip_model: str = "openai/clip-vit-base-patch16",
        sam_checkpoint: str = "/kaggle/working/sam_vit_b_01ec64.pth",
        img_size: int = 256,
    ):
        super().__init__()
        self.img_size = img_size

        # --- CLIP encoder (frozen) --- mirrors FakeShield CLIPVisionTower
        self.clip_encoder = CLIPVisionModel.from_pretrained(clip_model)
        self.clip_encoder.requires_grad_(False)
        self.clip_dim = self.clip_encoder.config.hidden_size      # 768

        # --- SAM backbone (encoder frozen, decoder trainable) ---
        self.sam = SAMBackbone(sam_checkpoint)

        # --- Detection head (simplified DTG) ---
        self.detection_head = DetectionHead(in_dim=self.clip_dim)

        # --- Feature projection (replaces TCM) ---
        self.feature_projection = FeatureProjection(
            in_dim=self.clip_dim, out_dim=256,
        )

        # Register SAM normalisation buffers
        self.register_buffer("sam_mean", self.SAM_MEAN)
        self.register_buffer("sam_std",  self.SAM_STD)

    # ------------------------------------------------------------------ #
    # helpers                                                              #
    # ------------------------------------------------------------------ #
    def _prepare_for_clip(self, images: torch.Tensor) -> torch.Tensor:
        """Resize to 224 for CLIP (ViT-B/16 native resolution)."""
        return F.interpolate(images, size=(224, 224), mode="bilinear",
                             align_corners=False)

    def _prepare_for_sam(self, images: torch.Tensor) -> torch.Tensor:
        """Denormalise ImageNet, renormalise for SAM, pad to 1024."""
        # Undo ImageNet normalisation from Albumentations
        imagenet_mean = torch.tensor([0.485, 0.456, 0.406],
                                     device=images.device).view(1, 3, 1, 1)
        imagenet_std  = torch.tensor([0.229, 0.224, 0.225],
                                     device=images.device).view(1, 3, 1, 1)
        x = images * imagenet_std + imagenet_mean              # [0,1]
        x = x * 255.0                                          # [0,255]

        # SAM normalisation
        x = (x - self.sam_mean) / self.sam_std

        # Resize longest side → 1024 then pad
        x = F.interpolate(x, size=(1024, 1024), mode="bilinear",
                          align_corners=False)
        return x

    # ------------------------------------------------------------------ #
    # forward                                                              #
    # ------------------------------------------------------------------ #
    def forward(self, images: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Args:
            images: (B, 3, H, W) ImageNet-normalised tensor from dataloader

        Returns dict:
            det_logits : (B,)         detection logits
            mask_logits: (B, H, W)    mask logits at input resolution
        """
        B, _, H, W = images.shape

        # 1) CLIP → global features
        clip_input = self._prepare_for_clip(images)
        with torch.no_grad():
            clip_out = self.clip_encoder(pixel_values=clip_input)
        cls_feat = clip_out.pooler_output                       # (B, 768)

        # 2) Detection (simplified DTG)
        det_logits = self.detection_head(cls_feat)              # (B,)

        # 3) Feature projection → SAM prompt (replaces TCM → h_<SEG>)
        h_prompt = self.feature_projection(cls_feat)            # (B, 256)

        # 4) SAM encoder (frozen)
        sam_input = self._prepare_for_sam(images)
        sam_feats = self.sam.encode_image(sam_input)             # (B, 256, 64, 64)

        # 5) SAM mask decoder (trainable)
        mask_low = self.sam.decode_mask(sam_feats, h_prompt)     # (B, 256, 256)

        # 6) Resize mask to input resolution
        mask_logits = F.interpolate(
            mask_low.unsqueeze(1), size=(H, W),
            mode="bilinear", align_corners=False,
        ).squeeze(1)                                            # (B, H, W)

        return {
            "det_logits":  det_logits,
            "mask_logits": mask_logits,
        }


print("FakeShieldLite model class defined.")

In [ ]:
# ============================================================================
# 5.6 Instantiate Model & Print Summary
# ============================================================================

model = FakeShieldLite(
    clip_model="openai/clip-vit-base-patch16",
    sam_checkpoint=SAM_CHECKPOINT,
    img_size=IMG_SIZE,
).to(DEVICE)

# Parameter counts
total_params    = sum(p.numel() for p in model.parameters())
trainable       = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen          = total_params - trainable

print(f"Model loaded on {DEVICE}")
print(f"  Total params     : {total_params:>12,}")
print(f"  Trainable params : {trainable:>12,}  ({100*trainable/total_params:.1f}%)")
print(f"  Frozen params    : {frozen:>12,}  ({100*frozen/total_params:.1f}%)")

# Quick forward-pass smoke test
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out = model(dummy)
    print(f"\nSmoke test passed:")
    print(f"  det_logits  shape: {out['det_logits'].shape}")
    print(f"  mask_logits shape: {out['mask_logits'].shape}")

if torch.cuda.is_available():
    alloc = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Peak GPU memory  : {alloc:.2f} GB")

---
## Section 6 — Loss Functions

FakeShield’s MFLM training loss (Eq. 5 in the paper):

```
ℓ_loc = ℓ_ce(ŷ_txt, y_txt) + α · ℓ_bce(Ŵ_loc, M_loc) + β · ℓ_dice(Ŵ_loc, M_loc)
```

FakeShield-Lite adapts this to:

```
ℓ_total = 1.0 · ℓ_det_bce  +  2.0 · ℓ_mask_bce  +  0.5 · ℓ_mask_dice
```

where `ℓ_det_bce` replaces the text cross-entropy (we do binary detection instead of text generation).

In [ ]:
# ============================================================================
# 6.1 Loss Functions
# ============================================================================

class DiceLoss(nn.Module):
    """Dice loss for binary segmentation. Matches FakeShield's calculate_dice_loss."""
    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = logits.sigmoid()
        p = probs.flatten(1)
        t = targets.flatten(1)
        intersection = 2.0 * (p * t).sum(dim=-1)
        union = p.sum(dim=-1) + t.sum(dim=-1)
        dice = 1.0 - (intersection + self.smooth) / (union + self.smooth)
        return dice.mean()


class CombinedLoss(nn.Module):
    """
    Combined loss matching FakeShield Eq. 5 structure:
      L = w_det * BCE(det) + w_bce * BCE(mask) + w_dice * Dice(mask)
    """
    def __init__(self, w_det=1.0, w_bce=2.0, w_dice=0.5):
        super().__init__()
        self.w_det  = w_det
        self.w_bce  = w_bce
        self.w_dice = w_dice
        self.det_bce  = nn.BCEWithLogitsLoss()
        self.mask_bce = nn.BCEWithLogitsLoss()
        self.mask_dice = DiceLoss()

    def forward(self, det_logits, mask_logits, labels, masks):
        # Detection loss (all samples)
        loss_det = self.det_bce(det_logits, labels)

        # Mask losses (only on tampered samples that have masks)
        tampered = labels > 0.5
        if tampered.any():
            m_pred = mask_logits[tampered]
            m_gt   = masks[tampered]
            loss_bce  = self.mask_bce(m_pred, m_gt)
            loss_dice = self.mask_dice(m_pred, m_gt)
        else:
            loss_bce  = torch.tensor(0.0, device=det_logits.device)
            loss_dice = torch.tensor(0.0, device=det_logits.device)

        total = (self.w_det  * loss_det
               + self.w_bce  * loss_bce
               + self.w_dice * loss_dice)

        return {
            "total":     total,
            "det_bce":   loss_det,
            "mask_bce":  loss_bce,
            "mask_dice": loss_dice,
        }


criterion = CombinedLoss(w_det=1.0, w_bce=2.0, w_dice=0.5)
print("CombinedLoss ready  (w_det=1.0, w_bce=2.0, w_dice=0.5)")

---
## Section 7 — Training Pipeline

In [ ]:
# ============================================================================
# 7.1 Optimiser & Scheduler
# ============================================================================

NUM_EPOCHS = 20
LR = 1e-4

# Differential learning rates (FakeShield uses lower LR for pretrained parts)
param_groups = [
    {"params": model.detection_head.parameters(),     "lr": LR,       "name": "det_head"},
    {"params": model.feature_projection.parameters(), "lr": LR,       "name": "feat_proj"},
    {"params": model.sam.mask_decoder.parameters(),   "lr": LR * 0.5, "name": "sam_dec"},
]

optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-6,
)

scaler = GradScaler()     # mixed-precision training

print(f"Optimiser : AdamW  (weight_decay=0.01)")
print(f"Scheduler : CosineAnnealing  (T_max={total_steps})")
print(f"Epochs    : {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Steps/ep  : {len(train_loader)}")

In [ ]:
# ============================================================================
# 7.2 Training Loop
# ============================================================================

history = {"train_loss": [], "val_loss": [],
           "val_det_f1": [], "val_iou": [], "val_dice": []}
best_val_iou = 0.0
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    model.clip_encoder.eval()           # keep frozen parts in eval
    model.sam.image_encoder.eval()

    running_loss = 0.0
    pbar = tqdm(loader, desc="  Train", leave=False)

    for batch in pbar:
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        with autocast():
            out = model(images)
            losses = criterion(out["det_logits"], out["mask_logits"],
                               labels, masks)

        scaler.scale(losses["total"]).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += losses["total"].item()
        pbar.set_postfix(loss=f"{losses['total'].item():.4f}")

    return running_loss / len(loader)


print("Training loop defined.")

---
## Section 8 — Evaluation Metrics

In [ ]:
# ============================================================================
# 8.1 Metric Functions
# ============================================================================

def compute_iou(pred: torch.Tensor, target: torch.Tensor) -> float:
    """IoU = |P ∩ G| / |P ∪ G|   (threshold at 0.5)."""
    p = pred.bool()
    t = target.bool()
    inter = (p & t).float().sum().item()
    union = (p | t).float().sum().item()
    return inter / union if union > 0 else (1.0 if inter == 0 else 0.0)


def compute_dice(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Dice = 2|P ∩ G| / (|P| + |G|)."""
    p = pred.bool()
    t = target.bool()
    inter = (p & t).float().sum().item()
    total = p.float().sum().item() + t.float().sum().item()
    return 2.0 * inter / total if total > 0 else (1.0 if inter == 0 else 0.0)


def compute_det_metrics(preds: np.ndarray, labels: np.ndarray) -> dict:
    """Accuracy, Precision, Recall, F1 for binary detection."""
    tp = ((preds == 1) & (labels == 1)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    eps = 1e-8
    acc  = (tp + tn) / (tp + tn + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    return {"accuracy": float(acc), "precision": float(prec),
            "recall": float(rec), "f1": float(f1)}


print("Metric functions: compute_iou, compute_dice, compute_det_metrics")

---
## Section 9 — Validation Loop

In [ ]:
# ============================================================================
# 9.1 Validation Function
# ============================================================================

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_det_preds, all_det_labels = [], []
    all_ious, all_dices = [], []

    for batch in tqdm(loader, desc="  Val", leave=False):
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)
        labels = batch["label"].to(device)

        with autocast():
            out = model(images)
            losses = criterion(out["det_logits"], out["mask_logits"],
                               labels, masks)

        running_loss += losses["total"].item()

        # Detection
        preds = (out["det_logits"].sigmoid() > 0.5).float()
        all_det_preds.extend(preds.cpu().numpy())
        all_det_labels.extend(labels.cpu().numpy())

        # Localization (tampered only)
        mask_pred = (out["mask_logits"].sigmoid() > 0.5).float()
        for i in range(len(labels)):
            if labels[i] > 0.5:
                all_ious.append(compute_iou(mask_pred[i], masks[i]))
                all_dices.append(compute_dice(mask_pred[i], masks[i]))

    det = compute_det_metrics(np.array(all_det_preds), np.array(all_det_labels))
    avg_iou  = float(np.mean(all_ious))  if all_ious  else 0.0
    avg_dice = float(np.mean(all_dices)) if all_dices else 0.0

    return {
        "loss": running_loss / len(loader),
        **det,
        "iou": avg_iou,
        "dice": avg_dice,
    }


print("Validation function defined.")

In [ ]:
# ============================================================================
# 9.2 Run Training
# ============================================================================

SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 50)

    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, DEVICE)

    val = validate(model, val_loader, criterion, DEVICE)

    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val["loss"])
    history["val_det_f1"].append(val["f1"])
    history["val_iou"].append(val["iou"])
    history["val_dice"].append(val["dice"])

    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val['loss']:.4f}")
    print(f"  Detection  \u2014 ACC: {val['accuracy']:.4f}  P: {val['precision']:.4f}  "
          f"R: {val['recall']:.4f}  F1: {val['f1']:.4f}")
    print(f"  Localization \u2014 IoU: {val['iou']:.4f}  Dice: {val['dice']:.4f}")

    # Save best model
    if val["iou"] > best_val_iou:
        best_val_iou = val["iou"]
        ckpt_path = os.path.join(SAVE_DIR, "best_model.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_metrics": val,
        }, ckpt_path)
        print(f"  >> New best model saved  (IoU={best_val_iou:.4f})")

print("\n" + "=" * 50)
print(f"Training complete.  Best val IoU: {best_val_iou:.4f}")

In [ ]:
# ============================================================================
# 9.3 Training Curves
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss
axes[0].plot(epochs_range, history["train_loss"], label="Train", color="tab:blue")
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   color="tab:orange")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Localization
axes[1].plot(epochs_range, history["val_iou"],  label="IoU",  color="tab:green")
axes[1].plot(epochs_range, history["val_dice"], label="Dice", color="tab:cyan")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_title("Localization"); axes[1].legend(); axes[1].grid(alpha=0.3)

# Detection F1
axes[2].plot(epochs_range, history["val_det_f1"], label="F1", color="tab:red")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1")
axes[2].set_title("Detection F1"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle("FakeShield-Lite vF.0 — Training Curves", fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 10 — Visualization of Predictions

In [ ]:
# ============================================================================
# 10.1 Load Best Checkpoint for Evaluation
# ============================================================================

ckpt = torch.load(os.path.join(SAVE_DIR, "best_model.pth"), map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {ckpt['epoch']+1}")
print(f"  Val metrics at save: {ckpt['val_metrics']}")

In [ ]:
# ============================================================================
# 10.2 Prediction Visualisation Grid
# ============================================================================

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def denormalise(tensor):
    """Convert ImageNet-normalised tensor back to uint8 numpy image."""
    img = tensor.cpu().permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img * 255, 0, 255).astype(np.uint8)


def visualize_predictions(model, dataset, device, n=8):
    """
    Show grid: Original | Ground Truth Mask | Predicted Mask | Overlay
    """
    model.eval()
    # Pick tampered samples for richer visualisation
    tampered_idx = [i for i in range(len(dataset))
                    if dataset.samples[i][2] == 1]
    indices = random.sample(tampered_idx, min(n, len(tampered_idx)))

    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    titles = ["Original Image", "Ground Truth", "Predicted Mask", "Overlay"]
    for ax, t in zip(axes[0], titles):
        ax.set_title(t, fontsize=13, fontweight="bold")

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        img_t = sample["image"].unsqueeze(0).to(device)

        with torch.no_grad(), autocast():
            out = model(img_t)

        det_score = out["det_logits"][0].sigmoid().item()
        mask_prob = out["mask_logits"][0].sigmoid().cpu().numpy()
        mask_pred = (mask_prob > 0.5).astype(np.float32)

        img_np = denormalise(sample["image"])
        gt_np  = sample["mask"].numpy()

        # Overlay: red tint on predicted tamper region
        overlay = img_np.copy().astype(np.float32)
        red = np.zeros_like(overlay); red[:, :, 0] = 255
        m3 = np.stack([mask_pred]*3, axis=-1)
        overlay = overlay * (1 - 0.4 * m3) + red * (0.4 * m3)
        overlay = np.clip(overlay, 0, 255).astype(np.uint8)

        axes[row, 0].imshow(img_np);                        axes[row, 0].axis("off")
        axes[row, 1].imshow(gt_np, cmap="hot", vmin=0, vmax=1); axes[row, 1].axis("off")
        axes[row, 2].imshow(mask_pred, cmap="hot", vmin=0, vmax=1); axes[row, 2].axis("off")
        axes[row, 3].imshow(overlay);                       axes[row, 3].axis("off")

        axes[row, 0].set_ylabel(f"det={det_score:.2f}", fontsize=11, rotation=0,
                                 labelpad=50)

    plt.suptitle("FakeShield-Lite vF.0 — Predictions", fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()


visualize_predictions(model, test_ds, DEVICE, n=6)

---
## Section 11 — Baseline Results

In [ ]:
# ============================================================================
# 11.1 Evaluate on Test Set
# ============================================================================

test_metrics = validate(model, test_loader, criterion, DEVICE)

print("\n" + "=" * 55)
print("   FakeShield-Lite vF.0 — Test Set Results")
print("=" * 55)
print(f"")
print(f"  DETECTION (image-level)")
print(f"  {'Accuracy':<12}: {test_metrics['accuracy']:.4f}")
print(f"  {'Precision':<12}: {test_metrics['precision']:.4f}")
print(f"  {'Recall':<12}: {test_metrics['recall']:.4f}")
print(f"  {'F1 Score':<12}: {test_metrics['f1']:.4f}")
print(f"")
print(f"  LOCALIZATION (pixel-level, tampered images)")
print(f"  {'IoU':<12}: {test_metrics['iou']:.4f}")
print(f"  {'Dice':<12}: {test_metrics['dice']:.4f}")
print(f"")
print("=" * 55)
print("  This is the vF.0 BASELINE. Improvements planned")
print("  in subsequent versions.")
print("=" * 55)

---
## Section 12 — Model Saving

In [ ]:
# ============================================================================
# 12.1 Save Final Weights
# ============================================================================

FINAL_PATH = "/kaggle/working/model_vF02_fakeshield_lite.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "test_metrics": test_metrics,
    "config": {
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "lr": LR,
    },
}, FINAL_PATH)

size_mb = os.path.getsize(FINAL_PATH) / 1e6
print(f"Model saved: {FINAL_PATH}  ({size_mb:.1f} MB)")
print("This file will appear in the Output tab of your Kaggle notebook.")

In [ ]:
# ============================================================================
# 12.2 Reload Model (demonstration)
# ============================================================================

# To reload in a fresh Kaggle session:
#
# model = FakeShieldLite(sam_checkpoint=SAM_CHECKPOINT).to(DEVICE)
# ckpt = torch.load("/kaggle/working/model_vF02_fakeshield_lite.pth", map_location=DEVICE)
# model.load_state_dict(ckpt["model_state_dict"])
# model.eval()

print("Reload code provided above (commented out).")

---
## Section 13 — Next Steps

This notebook is **vF.0.4 — Kaggle-native baseline**.

### Dataset: CASIA Splicing Detection + Localization
- Connected directly as Kaggle input (no download step)
- Corrected ground-truth from [SunnyHaze/CASIA2.0-Corrected-Groundtruth](https://github.com/SunnyHaze/CASIA2.0-Corrected-Groundtruth)
- ~7,491 authentic + ~5,123 tampered images with corrected masks

### Planned improvements for subsequent versions:

| Version | Improvement | Rationale |
|---|---|---|
| vF.1 | Larger input resolution (384) | Better fine-grained mask quality |
| vF.1 | Multi-dataset training (CASIA + Coverage + Columbia) | Improved generalisation |
| vF.2 | Robustness testing (JPEG, resize, noise) | Assignment bonus points |
| vF.2 | Ablation study (no SAM, no CLIP, no augmentation) | Demonstrates understanding |
| vF.3 | Better encoder (CLIP ViT-L or DINOv2) | Higher detection accuracy |
| vF.3 | Edge-aware loss | Better boundary delineation |

### Key limitations of this baseline:

1. **No text explanations** — FakeShield’s LLM explainability is removed
2. **SAM ViT-B** instead of ViT-H — lower segmentation quality
3. **Single dataset** — original FakeShield trains on PS + DF + AIGC data
4. **No domain tagging** — 3-class DTG simplified to binary
5. **Direct projection** instead of TCM — less sophisticated text-to-visual alignment

---

**End of vF.0.4 FakeShield-Lite Kaggle Baseline Notebook**